# YOLO11n — Plate Detector (Kaggle quickstart)

Notebook train nhanh YOLO11n cho detector biển số, dùng dataset thô (chỉ `data.yaml` + `images/` + `labels/`, KHÔNG có manifest.json như notebook `yolo11_plate.ipynb` gốc).

## Cách dùng
1. Upload file `vn_plate_detect_yolo11.zip` thành Kaggle Dataset (Add Data > Upload).
2. Trong Notebook Settings, bật **GPU** (T4 x2 hoặc P100) và **Internet** (để tải `yolo11n.pt`).
3. Attach dataset vừa upload vào notebook — sẽ xuất hiện ở `/kaggle/input/<dataset-slug>/`.
4. Sửa `DATASET_SLUG` ở cell dưới cho đúng tên dataset bạn upload.
5. Run All.

In [ ]:
import os
os.environ.setdefault("ULTRALYTICS_TELEMETRY", "False")
os.environ.setdefault("YOLO_VERBOSE", "False")
import torch
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")

In [ ]:
# Cài ultralytics >= 8.3 (Kaggle image có thể đã sẵn) + polars (cần cho
# ultralytics 8.4.x khi lưu checkpoint — thiếu sẽ crash ở cuối epoch 1 với
# ModuleNotFoundError: No module named 'polars').
!pip install -q -U ultralytics polars

In [ ]:
from pathlib import Path

DATASET_SLUG = "vn-plate-detect-yolo11"  # doi theo ten dataset ban upload
DATA_DIR = Path("/kaggle/input") / DATASET_SLUG
assert DATA_DIR.is_dir(), f"khong thay dataset tai {DATA_DIR} - kiem tra DATASET_SLUG / attach dataset"

data_yaml = DATA_DIR / "data.yaml"
assert data_yaml.is_file(), f"data.yaml khong co tai {data_yaml}"

# data.yaml dung duong dan tuyet doi cua local (D:/Work/...) - viet lai
# path cho khop /kaggle/input/... truoc khi train.
yaml_text = data_yaml.read_text(encoding="utf-8")
print("data.yaml goc:\n", yaml_text)

fixed_yaml = Path("/kaggle/working/data.yaml")
fixed_yaml.write_text(
    f"path: {DATA_DIR}\n"
    "train: images/train\n"
    "val: images/val\n"
    "nc: 1\n"
    "names: ['plate']\n",
    encoding="utf-8",
)
print("data.yaml da sua:\n", fixed_yaml.read_text())

In [ ]:
from ultralytics import YOLO

EPOCHS = int(os.environ.get("EPOCHS", "60"))
IMGSZ = int(os.environ.get("IMGSZ", "640"))
BATCH = int(os.environ.get("BATCH", "32"))  # Kaggle GPU (T4 16GB) chiu duoc batch lon hon may local 4GB

model = YOLO("yolo11n.pt")
results = model.train(
    data=str(fixed_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=15,
    device=0,
    workers=4,
    save=True,
    save_period=-1,
    cache=False,
    plots=True,
    project="/kaggle/working/runs",
    name="plate_yolo11n",
    exist_ok=True,
    seed=42,
    verbose=True,
)

In [ ]:
from pathlib import Path
best = Path("/kaggle/working/runs/plate_yolo11n/weights/best.pt")
assert best.is_file(), f"best.pt khong co tai {best}"

metrics = YOLO(str(best)).val(data=str(fixed_yaml), imgsz=IMGSZ, device=0)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## Tai best.pt ve may
Vao tab **Output** cua notebook Kaggle sau khi chay xong, download
`runs/plate_yolo11n/weights/best.pt`, dat vao `models/plate_best.pt` trong
repo local, roi sua `PLATE_MODEL_PATH` trong `app/config.py` neu doi ten file.